# Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Seed
seed = 42

# sklearn format
from sklearn import set_config
set_config(
    transform_output='pandas',  # `transform` and `fit_transform` returns pd.DataFrame.
    display='text',             # `pipeline` properly prints into the jupyter output cell.
)

# Summary

## Terminologies

### Data
$$
X =
\begin{bmatrix}
x_{1,1} & x_{1,2} & \cdots & x_{1,n} \\
x_{2,1} & x_{2,2} & \cdots & x_{2,n} \\
\vdots  & \vdots  & \ddots & \vdots  \\
x_{m,1} & x_{m,2} & \cdots & x_{m,n}
\end{bmatrix}
\in \mathbb{R}^{m \times n}
$$

### Weights and Bias
$$
w =
\begin{bmatrix}
w_1 \\
w_2 \\
\vdots \\
w_n
\end{bmatrix}
\in \mathbb{R}^{n \times 1}
, \quad
b = 
\begin{bmatrix}
b_1 \\
b_2 \\
\vdots \\
b_m
\end{bmatrix}
\in \mathbb{R}^{m \times 1}
$$

### Metrics
- $R^2 = (1 - \frac{RSS}{TSS})$
  - Coefficient of determination.
  - A ratio of $y$ that is explained by $x$.
  - $RSS = \sum{(y - \hat{y})^2}$
  - $TSS = \sum{(y - \bar{y})^2}$

## List

| Idx | Name | Prediction | Objective | Solver | Complexity | Class | Use Case |
|---:|---|---|---|---|---|---|---|
| 1 | OLS | $\hat{y} = Xw + b$ | $\displaystyle\min_{w,b} \left\|\hat{y} - y\right\|_2^2$ | SVD / QR / normal equation | $O(mn^2)$, assuming $m \geq n$ | `LinearRegression()` | Baseline |
| 2 | Ridge | $\hat{y} = Xw + b$ | $\displaystyle\min_{w,b} \left\|\hat{y} - y\right\|_2^2 + \alpha\left\|w\right\|_2^2$ | Cholesky, SVD, LSQR, etc. | $\approx O(mn^2)$ | `Ridge()` | Correlated features / reduce variance |
| 3 | Lasso | $\hat{y} = Xw + b$ | $\displaystyle\min_{w,b} \frac{1}{2m}\left\|\hat{y} - y\right\|_2^2 + \alpha\left\|w\right\|_1$ | Coordinate descent | $\approx O(kmn)$ | `Lasso()` | Feature selection / sparse weights |
| 4 | Elastic Net | $\hat{y} = Xw + b$ | $\displaystyle\min_{w,b} \frac{1}{2m}\left\|\hat{y} - y\right\|_2^2 + \alpha\rho\left\|w\right\|_1 + \frac{\alpha(1-\rho)}{2}\left\|w\right\|_2^2$ | Coordinate descent | $\approx O(kmn)$ | `ElasticNet()` | Correlated features + feature selection |
| 5 | Logistic Regression | $\displaystyle p(y=1\mid X)=\sigma(Xw+b)$ | $\displaystyle\min_{w,b}\;-\sum_i\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right] + \text{regularization}$ | L-BFGS, Newton, SAG, SAGA, etc. | $\approx O(kmn)$, solver-dependent | `LogisticRegression()` | Binary / multiclass classification |
| 6 | Polynomial Regression | $\hat{y} = \Phi(X)w+b$ | Same as the regression model fitted on $\Phi(X)$ | Feature expansion + linear-model solver | Depends on number of generated polynomial features | `PolynomialFeatures()` + regressor | Non-linear relationships / feature interactions |
| 7 | RANSAC | $\hat{y} = Xw+b$ | Maximize the number of inliers where $\left|y_i-\hat{y}_i\right| \leq \text{threshold}$ | Random sampling + consensus + refit | $\approx O\left(k(s n^2 + mn)\right)$ for OLS base estimator | `RANSACRegressor()` | Strong outliers / contaminated data |
| 8 | Huber Regression | $\hat{y} = Xw+b$ | $\displaystyle\min_{w,b,\sigma>0}\sum_{i=1}^{m}\sigma\left[H_{\epsilon}\left(\frac{r_i}{\sigma}\right)+1\right]+\alpha\lVert w\rVert_2^2$ | L-BFGS-B | Iterative; solver-dependent | `HuberRegressor()` | Outliers without completely removing samples |
| 9 | Quantile Regression | $\hat{y}_{\tau} = Xw+b$ | $\displaystyle\min_{w,b}\sum_i \rho_{\tau}(y_i-\hat{y}_i)+\alpha\lVert w\rVert_1$ | Linear programming | Solver-dependent | `QuantileRegressor()` | Conditional quantiles / asymmetric prediction |

# Data

In [2]:
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(
    return_X_y=True,
    as_frame=True,
)

# 1. Ordinary Least Square

1. Center: $x' = x - \bar{x}, \quad y' = y - \bar{y}$
2. Solve: $\displaystyle\min_{w, b} ||\hat{y} - y||_2^2$
3. Calculate the intercept: $b = \bar{y} - \bar{x}^Tw$

## 1) Basic

In [3]:
from sklearn.linear_model import LinearRegression

# Model
model = LinearRegression(
    fit_intercept=True,     # if False, intercept becomes 0.
    positive=False,         # if True, all weights become positive.
)

# Methods
model.fit(X, y, sample_weight=None)
model.predict(X)
model.score(X, y, sample_weight=None)   # R^2.
model.set_params(positive=True)

# Attributes
model.coef_                 # w.
model.intercept_            # b.
model.n_features_in_        # num of features
model.feature_names_in_     # feature names.
model.rank_                 # num of independent features.
model.singular_             # singular values of X. Only available when X is dense.

array([1.62693602e+05, 1.73178899e+03, 1.48489456e+03, 4.13011226e+02,
       3.68485454e+02, 2.47744430e+02, 7.18215739e+01, 2.50607596e+01])

## 2) Independent Features

In [4]:
n_features  = model.n_features_in_
rank        = model.rank_
singular    = model.singular_
k           = singular[0] / singular[-1]    # largest / smallest.

print(f"Independent features: {rank}/{n_features}")
print(f"Singular values: \n{singular}")
print(f"Condition number, k: {k}")          # large k means poor conditioning.

Independent features: 8/8
Singular values: 
[1.62693602e+05 1.73178899e+03 1.48489456e+03 4.13011226e+02
 3.68485454e+02 2.47744430e+02 7.18215739e+01 2.50607596e+01]
Condition number, k: 6491.966124157369


# 2. Regularization

## 1) Ridge

- Objective
  - $\displaystyle\min_{w, b} ||\hat{y} - y||_2^2 + \alpha ||w||_2^2$
- Solver
  - `lbfgs`: if `positive=True`.
  - `cholesky`: if X is not sparse.
  - `sparse_cg`: both are not fulfilled.

### Basic

In [5]:
from sklearn.linear_model import Ridge

# Model
model = Ridge(
    alpha=1,
    solver='auto',
    max_iter=None,
    tol=1e-4,
    fit_intercept=True,
    positive=False,
    random_state=seed,
)

# Methods
model.fit(X, y, sample_weight=None)
model.predict(X)
model.score(X, y, sample_weight=None)       # R^2.
model.set_params(positive=True)

# Attributes
model.coef_
model.intercept_
model.n_features_in_
model.feature_names_in_
model.n_iter_
model.solver_

'cholesky'

### `RidgeClassifier`

In [6]:
from sklearn.linear_model import RidgeClassifier

model = RidgeClassifier(...)    # same as Ridge().

### `RidgeCV`

In [7]:
from sklearn.linear_model import RidgeCV, RidgeClassifierCV

# Model
model = RidgeCV(
    cv=None,
    gcv_mode=None,
    store_cv_results=True,      # default=None.
    alpha_per_target=False,     # if True, learn alpha by each target in multi-output problem.
)

# Methods (same)
model.fit(X, y)

# Attributes
model.cv_results_       # only available when `cv=None` and `store_cv_results=True`.
model.alpha_
model.best_score_
model.n_features_in_
model.feature_names_in_

array(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population',
       'AveOccup', 'Latitude', 'Longitude'], dtype=object)

## 2) Lasso

In [8]:
from sklearn.linear_model import Lasso, LassoCV

model = Lasso()      # same as Ridge.
model = LassoCV()    # same as Ridge.

## 3) Elastic Net

In [9]:
from sklearn.linear_model import ElasticNet, ElasticNetCV

model = ElasticNet(
    l1_ratio=0.5,       # Lasso ratio.
)

model = ElasticNetCV(
    l1_ratio=0.5,
)

# 3. Logistic Regression

- Apply sigmoid function on the linear combination.
  - $p(z) = \frac{1}{1+e^{-z}}$, where $z = Xw + b$
  - $p \lt 0.5 \rightarrow negative$
  - $p \geq 0.5 \rightarrow positive$
- Objective: Cross entropy

In [10]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

# Sample data
y_clf = y > y.mean()

# Model
model = LogisticRegression(
    l1_ratio=0.0,
    C=1.0,                  # 1 / alpha.
    class_weight=None,      # 'balanced': 1 / class_ratio.
    max_iter=10_000,
    warm_start=False,
    random_state=seed,
)

# Methods
model.fit(X, y_clf, sample_weight=None)
model.score(X, y_clf, sample_weight=None)   # return accruacy.

model.predict(X)
model.predict_proba(X)          # predicts probability.
model.predict_log_proba(X)      # log(pr).
model.decision_function(X)      # confidence scores, (m, n).

# Attributes
model.coef_
model.intercept_
model.classes_
model.n_features_in_
model.feature_names_in_
model.n_iter_

array([1837], dtype=int32)

# 4. Polynomial Regression

In [11]:
from sklearn.preprocessing import PolynomialFeatures

# Polynomial features
poly = PolynomialFeatures(
    degree=2,
    interaction_only=False,     # if True, exclude non-unique features, e.g. x1**2.
    include_bias=True,
)

# Methods
X_poly = poly.fit_transform(X)

# Attributes
poly.powers_                # matrix for combination, e.g. x3 = x1*x2 -> [1, 1].
poly.n_features_in_
poly.feature_names_in_
poly.n_output_features_

45

# 5. Robust Regression

## 1) RANSAC: Random Sample Consensus

1. Select `min_samples` random samples.
2. Check `is_data_valid`.
3. Fit and check `is_model_valid`.
4. Select inliners: $|\hat{y_i} - y_i| \leq $ `residual_threshold`.
5. Largest `n_inliners` -> best model.

In [12]:
from sklearn.linear_model import RANSACRegressor, LinearRegression

# Model
estimator = LinearRegression()
model = RANSACRegressor(
    estimator=estimator,
    min_samples=0.5,            # better to specify manually.
    is_data_valid=None,         # if returns False, the subset is skipped.
    is_model_valid=None,        # if returns False, the subset is skipped.

    max_trials=100,
    max_skips=np.inf,
    stop_n_inliers=np.inf,
    stop_score=np.inf,
    stop_probability=0.99,

    residual_threshold=None,    # samples where '|y - y_hat| <= threshold' are inliner.
    loss='absolute_error',      # or 'squared_error'.

    random_state=seed,
)

## 2) Huber Regression

-  Huber loss
$$
H_{\epsilon}(z) =
\begin{cases}
z^2, & |z| \leq \epsilon \\[4pt]
2\epsilon |z| - \epsilon^2, & |z| > \epsilon
\end{cases}
$$

$$
\text{where } r = y - \hat{y}, \quad z = \frac{r}{\sigma}
$$

- Objective
$$
\min_{w,b,\sigma>0}
\sum_{i=1}^{m}
\sigma
\left[
H_{\epsilon}\left(\frac{r_i}{\sigma}\right) + 1
\right]
+
\alpha \lVert w \rVert_2^2
$$

In [13]:
from sklearn.linear_model import HuberRegressor

# Model
model = HuberRegressor(
    epsilon=1.35,           # threshold for outlier.
    alpha=1e-4,             # for l2 penalty.
    max_iter=1_000,
    warm_start=False,       # `.fit()` starts with previous fitted coefs.
    tol=1e-5,

    fit_intercept=True,
)

# Methods
model.fit(X, y, sample_weight=None)

# Attributes
model.scale_                # fitted sigma.

np.float64(0.393843158526704)

# 6. Quantile Regression

In [14]:
from sklearn.linear_model import QuantileRegressor

# Model
model = QuantileRegressor(
    quantile=0.5,           # if 0.75 -> 75% quantile (top 25%) -> 75% of y will be smaller than y_hat.
    alpha=1,                # for L1 penalty.
    fit_intercept=True,
    solver='warn',
    solver_options=None,
)